In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [2]:
# llm Setup 
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

In [3]:
# --- Prompts ---

draft_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a creative social media marketer. "
        "Write a tweet for the following product."
    ),
    (
        "human",
        "Product: {product_name}. "
        "Target Audience: {audience}."
    )
])

edit_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a senior editor. Your job is to rewrite content "
        "to sound more human and punchy. Remove emojis if they are excessive."
    ),
    (
        "human",
        "Here is the draft tweet:\n\n"
        "{draft_content}\n\n"
        "Make it better."
    )
])

translate_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a professional translator. "
        "Translate the following text into {language}. "
        "Maintain the slang and tone."
    ),
    (
        "human",
        "Text: {polished_content}"
    )
])

In [ ]:
# --- Chains ---

draft_chain = draft_prompt | llm | StrOutputParser()
edit_chain = edit_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()


# --- The Fixed Pipeline ---
# Usage of .assign ensures we keep 'language' from the initial input

final_chain = (
    RunnablePassthrough.assign(draft_content=draft_chain)
    | 
    RunnablePassthrough.assign(polished_content=edit_chain)
    | 
    RunnablePassthrough.assign(final_translation=translate_chain)
)

In [5]:
# --- Execution ---

inputs = {
    "product_name": "Eco-Friendly Bamboo Toothbrush",
    "audience": "Eco-conscious millennials",
    "language": "Spanish"
}

# This will now work because 'language' flows all the way to the end
result = final_chain.invoke(inputs)

print(f"--- 1. Initial Draft ---\n{result['draft_content']}\n")
print(f"--- 2. Polished Version ---\n{result['polished_content']}\n")
print(f"--- 3. Final Translation ---\n{result['final_translation']}")

--- 1. Initial Draft ---
🌍✨ Make a small change for a big impact! Switch to our eco-friendly bamboo toothbrush and join the movement towards a greener planet. Every brush counts! 🪥💚 #SustainableLiving #EcoFriendly #BambooToothbrush #MillennialGreen

--- 2. Polished Version ---
Make a simple switch for a huge impact! Try our eco-friendly bamboo toothbrush and be part of the movement for a greener planet. Every brush matters! #SustainableLiving #EcoFriendly #BambooToothbrush

--- 3. Final Translation ---
¡Haz un cambio sencillo para un impacto enorme! Prueba nuestra cepillo de dientes de bambú ecológico y sé parte del movimiento por un planeta más verde. ¡Cada cepillada cuenta! #VidaSostenible #Ecológico #CepilloDeDientesDeBambú
